# Redução de Dimensionalidade das Métricas Locais via PCA

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [2]:
DATASET_PATH = "../data/processed/dataset_features.parquet"

df = pd.read_parquet(DATASET_PATH)
df.shape


(165, 393)

In [4]:
df.head()

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
0,39449.859168,39102.152342,40387.630997,37566.960373,39408.112994,38719.391893,40815.072973,41275.499305,39649.516506,40196.211832,...,"#CODE:296.26 #DESC:Major Depressive Disorder,...",None,past subclinical dperession,0.0,7.0,1.0,7.0,29.5,No,No
1,41513.435372,41327.513954,40750.435276,40491.263384,41233.158432,41146.017954,40730.430082,41351.132165,39371.224746,39566.056291,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
2,40727.834359,40001.600624,40267.149863,39308.151269,39753.137398,39336.794272,40117.319684,40446.206489,38441.008195,38531.272048,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
3,38401.305380,38333.117784,37902.877026,37959.206795,38497.084816,37761.467224,37754.236560,37956.444283,36880.581031,37203.264413,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes
4,42558.276311,42950.141781,42023.976342,40857.941940,40478.291440,41090.310476,42519.571111,42339.443250,40755.136885,42859.521462,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes


### Seleção Métricas Locais

In [5]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

len(NODE_FEATURES)


366

In [7]:
X = df[NODE_FEATURES].values
y = df["condition"].values

### Padronização das Feaures

Substituindo Nan por zero:

In [21]:
X = np.nan_to_num(X, nan=0.0)

In [22]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [23]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([-1.99705572e-15, -1.66466167e-15, -1.47491447e-15,  1.61486985e-16,
         9.66230463e-16]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [24]:
np.isnan(X).sum()


np.int64(0)

In [25]:
nan_per_feature = pd.Series(np.isnan(X).sum(axis=0), index=NODE_FEATURES)
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [26]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)


In [27]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})


In [28]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()


Observa-se que aproximadamente 14 componentes são suficientes para explicar 80% da variância total.

### PCA com 2 componentes

In [29]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y


In [30]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()


Aqui você responde visualmente:

existe separação EO × EC?

existe sobreposição?

clusters?

### Contribuição das features

In [31]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)


In [32]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
weighted_degree_46,0.082558,0.017579
degree_46,0.082558,0.017579
weighted_degree_16,0.082146,0.031428
degree_16,0.082146,0.031428
weighted_degree_6,0.082016,0.028846
degree_6,0.082016,0.028846
weighted_degree_10,0.081638,0.019646
degree_10,0.081638,0.019646
weighted_degree_52,0.081264,0.048791
degree_52,0.081264,0.048791


In [33]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
clustering_20,0.056906,0.150630
local_efficiency_59,0.025072,0.142863
degree_59,0.030978,0.142661
weighted_degree_59,0.030978,0.142661
clustering_59,0.031085,0.135050
local_efficiency_60,0.021359,0.124430
degree_60,0.025563,0.122206
weighted_degree_60,0.025563,0.122206
clustering_40,0.070889,0.120346
clustering_15,0.071363,0.120328


Quais métricas e nós mais contribuem para cada componente?

### PCA com mais componentes - Para ML

In [34]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(165, 29)

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional indica padrões de organização distintos entre as
condições experimentais, embora com sobreposição parcial, sugerindo que métodos
não lineares ou classificadores supervisionados podem capturar diferenças mais sutis.

Os loadings indicam que métricas associadas a determinados nós apresentam maior
contribuição para os componentes principais, reforçando a relevância da análise
local da conectividade funcional.
